In [7]:
import pandas as pd
import numpy as np

train_path = "/content/train.csv"
test_path = "/content/test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

train_df.head()

,id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality,emotional_state,intensity
0,1,The ocean ambience helped me stop drifting and...,ocean,12,6.5,4,2,afternoon,mixed,calm_face,clear,focused,3
1,2,"I tried to relax during the forest ambience, y...",forest,35,6.0,2,4,evening,calm,tired_face,vague,restless,3
2,3,The forest session slowed my thoughts and I fe...,forest,3,NaN,2,1,night,overwhelmed,happy_face,clear,calm,3
3,4,"the mountain ambience was pleasant, though i c...",mountain,25,7.0,4,4,night,focused,calm_face,vague,neutral,1
4,5,"The rain session gave me a pause, but the pres...",rain,25,5.0,3,5,afternoon,NaN,tense_face,clear,overwhelmed,5


In [8]:
train_df = train_df.copy()
test_df = test_df.copy()

for col in train_df.columns:
    if train_df[col].dtype == "object":
        train_df[col] = train_df[col].fillna("unknown")
    else:
        train_df[col] = train_df[col].fillna(train_df[col].median())

for col in test_df.columns:
    if test_df[col].dtype == "object":
        test_df[col] = test_df[col].fillna("unknown")
    else:
        test_df[col] = test_df[col].fillna(train_df[col].median())

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=300)

X_text_train = tfidf.fit_transform(train_df["journal_text"])
X_text_test = tfidf.transform(test_df["journal_text"])

In [10]:
from sklearn.preprocessing import LabelEncoder

cat_cols = [
    "ambience_type",
    "time_of_day",
    "previous_day_mood",
    "face_emotion_hint",
    "reflection_quality"
]

encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col] = le.transform(test_df[col])
    encoders[col] = le

In [11]:
num_cols = [
    "duration_min",
    "sleep_hours",
    "energy_level",
    "stress_level"
]

X_num_train = train_df[cat_cols + num_cols].values
X_num_test = test_df[cat_cols + num_cols].values

In [12]:
from scipy.sparse import hstack

X_train = hstack([X_text_train, X_num_train])
X_test = hstack([X_text_test, X_num_test])

In [13]:
y_state = train_df["emotional_state"]
y_intensity = train_df["intensity"]

In [14]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

state_model = RandomForestClassifier(n_estimators=200, random_state=42)
state_model.fit(X_train, y_state)

intensity_model = RandomForestRegressor(n_estimators=200, random_state=42)
intensity_model.fit(X_train, y_intensity)

RandomForestRegressor(n_estimators=200, random_state=42)

In [15]:
state_pred = state_model.predict(X_test)
state_proba = state_model.predict_proba(X_test)

import numpy as np

intensity_pred = intensity_model.predict(X_test)
intensity_pred = np.clip(np.round(intensity_pred), 1, 5)

In [16]:
confidence = state_proba.max(axis=1)
uncertain_flag = (confidence < 0.6).astype(int)

In [17]:
original_time = test_df["time_of_day"].copy()

In [18]:
def decide_action(row, state, intensity):
    stress = row["stress_level"]
    energy = row["energy_level"]
    time = row["time_of_day"]

    if stress > 7 and energy < 4:
        return "rest", "now"

    if energy > 7 and time == 0:
        return "deep_work", "now"

    if stress > 5:
        return "breathing", "within_15_min"

    if energy < 3:
        return "rest", "tonight"

    return "light_planning", "later_today"


actions = []
times = []

for i in range(len(test_df)):
    a, t = decide_action(test_df.iloc[i], state_pred[i], intensity_pred[i])
    actions.append(a)
    times.append(t)

In [19]:
output = pd.DataFrame({
    "id": test_df["id"],
    "predicted_state": state_pred,
    "predicted_intensity": intensity_pred,
    "confidence": confidence,
    "uncertain_flag": uncertain_flag,
    "what_to_do": actions,
    "when_to_do": times
})

output.to_csv("predictions.csv", index=False)

output.head()

,id,predicted_state,predicted_intensity,confidence,uncertain_flag,what_to_do,when_to_do
0,10001,focused,3.0,0.515,1,light_planning,later_today
1,10002,restless,4.0,0.300,1,rest,tonight
2,10003,focused,3.0,0.320,1,rest,tonight
3,10004,focused,3.0,0.710,0,rest,tonight
4,10005,neutral,3.0,0.315,1,rest,tonight


In [20]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_state, test_size=0.2, random_state=42)

temp_model = RandomForestClassifier(n_estimators=200, random_state=42)
temp_model.fit(X_tr, y_tr)

val_pred = temp_model.predict(X_val)

errors = []

for i in range(len(val_pred)):
    if val_pred[i] != y_val.iloc[i]:
        errors.append(i)

len(errors)

95

In [21]:
for i in errors[:10]:
    print("TEXT:", train_df.iloc[i]["journal_text"])
    print("ACTUAL:", y_val.iloc[i])
    print("PREDICTED:", val_pred[i])
    print("STRESS:", train_df.iloc[i]["stress_level"])
    print("ENERGY:", train_df.iloc[i]["energy_level"])
    print("TIME:", train_df.iloc[i]["time_of_day"])
    print("----")

TEXT: after the forest track i feel peaceful and less pulled in every direction. my shoulders feel less tense.
ACTUAL: focused
PREDICTED: restless
STRESS: 2
ENERGY: 3
TIME: 3
----
TEXT: The rain sounds were nice, but I still feel unsettled and fidgety. Part of me wants to do everything at once.
ACTUAL: calm
PREDICTED: overwhelmed
STRESS: 3
ENERGY: 3
TIME: 0
----
TEXT: The forest session made me calmer, but part of me still feels uneasy. I feel better and not better at the same time. I can't tell if I need rest or momentum.
ACTUAL: calm
PREDICTED: overwhelmed
STRESS: 3
ENERGY: 2
TIME: 4
----
TEXT: The mountain session gave me a pause, but the pressure is still sitting hard on me. I'm carrying too much in my head.
ACTUAL: mixed
PREDICTED: overwhelmed
STRESS: 5
ENERGY: 3
TIME: 1
----
TEXT: I feel mentally clear after the mountain session and ready to tackle one thing at a time. I should begin with the hardest task first.
ACTUAL: focused
PREDICTED: overwhelmed
STRESS: 2
ENERGY: 3
TIME: 4
-